In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd
from pathlib import Path

from PIL import Image
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR = Path.cwd().parent.parent.parent.resolve() / "data"
metadata_dir = DATA_DIR / "GF1_B02_B03_B04_B08_Mask"
# Set GF1_RAW_DIR on the server when source scenes live outside this folder.
RAW_GF1_DIR = Path(os.environ.get("GF1_RAW_DIR", metadata_dir))
metadata_path = metadata_dir / "metadata.csv"

assert metadata_path.exists()

In [ ]:
df = pd.read_csv(metadata_path)
df.head()

In [ ]:
metadata = list(metadata_dir.glob("*.tif"))
metadata

In [ ]:
def resolve_source_path(row):
    filename = f"{row['filename']}.tif"
    candidates = [Path(str(row.get('path', ''))), RAW_GF1_DIR / filename, metadata_dir / filename]
    return next((str(path) for path in candidates if path.exists()), None)

df['path'] = df.apply(resolve_source_path, axis=1)
df['sensor'] = 'GF1'
missing_sources = df[df['path'].isna()]['filename'].tolist()
if missing_sources:
    print(f'Unavailable GF1 source scenes: {len(missing_sources)}')
df = df.dropna(subset=['path']).copy()
df.head(100)

In [ ]:
CSV_OUT_DIR = DATA_DIR / "metadata"
IMG_OUT_DIR = DATA_DIR / "images"
LABEL_OUT_DIR = DATA_DIR / "labels"
PRED_OUT_DIR = DATA_DIR / "predictions"

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
root_dir = notebook_dir.parent.parent
sys.path.append(str(root_dir))

root_dir

# Generate training chips

In [ ]:
training_targets_txt = CSV_OUT_DIR / 'training_targets_all.txt'
with open(training_targets_txt, 'r', encoding='utf-8') as f:
    training_targets = [line.strip() for line in f if line.strip().startswith('GF1_')]
print(f'GF1 targets in all-target list: {len(training_targets)}')
training_targets

In [ ]:
df_for_training = df[df['filename'].isin(training_targets)].copy()
unavailable_targets = sorted(set(training_targets) - set(df_for_training['filename']))
if unavailable_targets:
    print(f'GF1 targets not available on this machine: {len(unavailable_targets)}')
print(f'GF1 scenes to tile: {len(df_for_training)}')
df_for_training

In [ ]:
from benchmark.core.geotiff_tiler import GeoTIFFTiler
from benchmark.inference import predict_full_geotiff
import traceback

In [ ]:
# GeoTIFFTiler reads source bands 1/2/3 only and maps source band 5 to
# binary labels: cloud=1, background/shadow=0. It also writes BGR P2/P98
# statistics into every metadata row.
chip_frames = []
try:
    for i in range(len(df_for_training)):
        t = GeoTIFFTiler(df_row=df_for_training.iloc[i], display_thumbnail=False)
        chip_frames.append(t.get_chips(CSV_OUT_DIR, IMG_OUT_DIR, LABEL_OUT_DIR))
    if chip_frames:
        gf1_metadata = pd.concat(chip_frames, ignore_index=True)
        gf1_metadata.to_csv(CSV_OUT_DIR / 'GF1_binary_metadata.csv', index=False)
except Exception:
    traceback.print_exc()

# Generate testing chips

In [ ]:
testing_targets_txt = CSV_OUT_DIR / 'testing_targets_all.txt'
with open(testing_targets_txt, 'r', encoding='utf-8') as f:
    testing_targets = [
        line.strip() for line in f
        if line.strip().startswith('GF1_') and not line.lstrip().startswith('#')
    ]
print(f'Active GF1 testing targets: {len(testing_targets)}')
testing_targets

In [ ]:
df_for_testing = df[df['filename'].isin(testing_targets)].copy()
unavailable_test_targets = sorted(set(testing_targets) - set(df_for_testing['filename']))
if unavailable_test_targets:
    print(f'GF1 test targets not available on this machine: {len(unavailable_test_targets)}')
print(f'GF1 test scenes to tile: {len(df_for_testing)}')
df_for_testing

In [ ]:
from benchmark.configs import Configs

try:
    config = Configs.balanced()
    config.batch_size = 16
    config.use_tta = True
    model_weights_path = Path(config.model_name) / Path("assets/cloud_model.pt")
    test_chip_frames = []
    for i in range(len(df_for_testing)):
        t = GeoTIFFTiler(df_row=df_for_testing.iloc[i], display_thumbnail=False)
        test_chip_frames.append(t.get_chips(CSV_OUT_DIR, IMG_OUT_DIR, LABEL_OUT_DIR))
        predict_full_geotiff(t, PRED_OUT_DIR, model_weights_path=model_weights_path, config=config)
    if test_chip_frames:
        gf1_test_metadata = pd.concat(test_chip_frames, ignore_index=True)
        gf1_test_metadata.to_csv(CSV_OUT_DIR / 'GF1_binary_test_metadata.csv', index=False)
except Exception:
    traceback.print_exc()

In [ ]:
data_dir = DATA_DIR / "GF1_B02_B03_B04_B08_Mask"
data_tif_files = list(data_dir.glob("*.tif"))
data_tif_files

In [ ]:
pred_dir = DATA_DIR / "predictions"
pred_tif_files = list(pred_dir.glob("*_PredictedMask.tif"))
pred_tif_files

In [ ]:
from benchmark.visualization import display_thumbnail_with_prediction

try:
    for data_tif_file in data_tif_files:
        for pred_tif_file in pred_tif_files:
            filename = str(data_tif_file.stem)
            if filename in str(pred_tif_file) and filename in testing_targets:
                display_thumbnail_with_prediction(data_tif_file, pred_tif_file)
except Exception as e:
    traceback.print_exc()